In [1]:
# ==============================================================================
# SHIELD AGRO | PRECISION CACAO INTELLIGENCE 🛡️
# Versión: Robusta 2.0 - Esmeraldas & Manabí
# Análisis: Monilia, Phytophthora y GDA en tiempo real
# ==============================================================================

import ee
import geemap
import folium
from folium import plugins
import pandas as pd
import requests
from google.colab import auth

# 1. AUTENTICACIÓN Y CONFIGURACIÓN
auth.authenticate_user()
ee.Authenticate()
PROJECT_ID = 'shield-agro-iot' # REEMPLAZA CON TU PROYECTO ID

try:
    ee.Initialize(project=PROJECT_ID)
    print(f"✅ Shield-Agro conectado a Google Earth Engine")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

# 2. FUNCIÓN PARA OBTENER CLIMA REAL Y CALCULAR GDA (Open-Meteo)
def get_agrometeo_data(lat, lon):
    # Solicitamos temperatura actual y humedad de las últimas 24 horas
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true&hourly=relative_humidity_2m"
    try:
        r = requests.get(url).json()
        temp_actual = r['current_weather']['temperature']
        hr_actual = r['hourly']['relative_humidity_2m'][0]

        # Cálculo de GDA (Grados Días Acumulados)
        # Temp Base Cacao = 18.5°C (Promedio para desarrollo de patógenos)
        temp_base = 18.5
        gda_hoy = max(0, temp_actual - temp_base)
        gda_acumulado = gda_hoy * 14 # Simulamos acumulado de 14 días para el DemoDay

        return temp_actual, hr_actual, gda_acumulado
    except:
        return 26.5, 88.0, 112.5 # Valores por defecto en caso de error de red

# 3. PROCESAMIENTO SATELITAL EN GEE (Riesgo Fitosanitario)
# Definimos áreas de estudio
esmeraldas = ee.Geometry.Rectangle([-80.2, 0.1, -78.8, 1.3])
manabi = ee.Geometry.Rectangle([-81.2, -1.8, -79.3, 0.6])

# Filtramos Sentinel-2 (Último mes)
s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
      .filterDate('2026-04-01', '2026-05-12')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
      .median())

# Cálculo de Índices
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI') # Humedad en vegetación

# Modelo Lógico de Riesgo (Inverso al vigor de la planta)
riesgo_base = ndvi.multiply(-1).add(1).clip(esmeraldas.union(manabi))

# 4. CREAR MAPA BASE (OpenStreetMap)
m = folium.Map(location=[0.1, -79.5], zoom_start=8, tiles='OpenStreetMap')

# 5. AÑADIR CAPA DE RIESGO GEE
map_id = riesgo_base.getMapId({'min': 0.2, 'max': 0.8, 'palette': ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']})
folium.TileLayer(
    tiles=map_id['tile_fetcher'].url_format,
    attr='Shield-Agro Satellite Data',
    name='Mapa de Calor de Riesgo (GEE)',
    overlay=True,
    opacity=0.6
).add_to(m)

# 6. DATOS DE SENSORES Y TRAZABILIDAD (Puntos de Control)
sensores = pd.DataFrame({
    'lote': ['Lote A-1 (Esmeraldas)', 'Lote M-2 (Manabí)'],
    'lat': [0.90, -0.60],
    'lon': [-79.65, -80.10],
    'variedad': ['Nacional Arriba', 'CCN-51'],
    'encargado': ['Gabriela Peña', 'Equipo Técnico Shield'],
    'ultima_poda': ['05-May-2026', '28-Abr-2026']
})

# 7. GENERAR MARCADORES INTERACTIVOS CON TRAZABILIDAD
for i, row in sensores.iterrows():
    temp, hr, gda = get_agrometeo_data(row['lat'], row['lon'])

    # Lógica de Riesgo Específico para el Popup
    riesgo_monilia = "ALTO" if hr > 85 and temp > 22 else "MODERADO"
    riesgo_phyto = "CRÍTICO" if hr > 90 and temp < 24 else "BAJO"

    # Diseño HTML del Cuadro de Trazabilidad
    color_header = "#c0392b" if riesgo_monilia == "ALTO" else "#1e8449"

    html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; width: 260px; border-radius: 12px; overflow: hidden; border: 1px solid #ddd;">
        <div style="background-color: {color_header}; color: white; padding: 12px; text-align: center;">
            <h3 style="margin: 0; font-size: 14px;">🛡️ TRAZABILIDAD SHIELD-AGRO</h3>
            <small>ID: {row['lote']}</small>
        </div>
        <div style="padding: 15px; font-size: 12px; background: #fff;">
            <b>📊 Parámetros Agrometeorológicos:</b><br>
            🌡️ Temp: {temp}°C | 💧 HR: {hr}%<br>
            📈 <b>GDA Acumulados:</b> {gda:.1f} °D<br>
            <hr style="margin: 10px 0; border: 0.5px solid #eee;">
            <b>🍄 Riesgo de Patógenos:</b><br>
            • Monilia: <span style="color:orange;">{riesgo_monilia}</span><br>
            • Phytophthora: <span style="color:red;">{riesgo_phyto}</span><br>
            <hr style="margin: 10px 0; border: 0.5px solid #eee;">
            <b>🌱 Información de Campo:</b><br>
            Variedad: {row['variedad']}<br>
            Poda: {row['ultima_poda']}<br>
            Resp: {row['encargado']}
        </div>
        <div style="background: #f4f4f4; padding: 8px; text-align: center;">
            <button style="border: none; background: #2c3e50; color: white; padding: 5px 10px; border-radius: 4px; font-size: 10px;">GENERAR REPORTE PDF</button>
        </div>
    </div>
    """

    # Crear marcador de trazabilidad (Polígono Diamante)
    folium.RegularPolygonMarker(
        location=[row['lat'], row['lon']],
        number_of_sides=4,
        radius=14,
        rotation=45,
        color='#2c3e50',
        fill_color=color_header,
        fill_opacity=0.8,
        popup=folium.Popup(html, max_width=280),
        tooltip="Click para Trazabilidad Completa"
    ).add_to(m)

# 8. ELEMENTOS FINALES Y LEYENDA
folium.LayerControl().add_to(m)

# Guardar el mapa
m.save("index.html")
print("✅ Proceso completado. Mapa interactivo robusto generado como 'index.html'.")

# Visualizar en Colab
m



✅ Shield-Agro conectado a Google Earth Engine


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/S2_SR_HARMONIZED

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


✅ Proceso completado. Mapa interactivo robusto generado como 'index.html'.


In [1]:
# ==============================================================================
# SHIELD AGRO | PRECISION CACAO INTELLIGENCE 🛡️
# Versión: Pro-Satelital Robusta (GEE + Open-Meteo + Folium)
# ==============================================================================

import ee, folium, requests
from folium import plugins
import pandas as pd
from google.colab import auth

# 1. AUTENTICACIÓN
auth.authenticate_user()
ee.Authenticate()
ee.Initialize(project='shield-agro-iot')

# 2. CONFIGURACIÓN DE DATOS (6 LOTES ESTRATÉGICOS)
sensores_pro = pd.DataFrame({
    'lote': ['Esmeraldas-Norte', 'Muisne-Costa', 'Quinindé-Agro', 'Chone-Centro', 'Portoviejo-Valle', 'El Carmen-Norte'],
    'lat': [1.15, 0.55, 0.32, -0.68, -1.02, -0.27],
    'lon': [-78.85, -80.02, -79.45, -80.08, -80.45, -79.46],
    'variedad': ['Nacional', 'CCN-51', 'CCN-51', 'Nacional', 'Híbrido', 'CCN-51'],
    'responsable': ['Ing. Peña', 'Téc. Lucas', 'Ing. Gabriel', 'Téc. Eduardo', 'Ing. Estefanía', 'Equipo Shield']
})

# 3. PROCESAMIENTO SATELITAL (Look & Feel de Google Earth Engine)
def get_gee_layer():
    # Sentinel-2 con corrección atmosférica
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR")
          .filterDate('2026-03-01', '2026-05-12')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .median())

    ndvi = s2.normalizedDifference(['B8', 'B4'])
    # Paleta característica de riesgo: Verde (Sano) -> Rojo (Riesgo)
    vis_params = {
        'min': 0.2, 'max': 0.8,
        'palette': ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
    }
    return ndvi.getMapId(vis_params)

# 4. FUNCIÓN CLIMÁTICA PARA PREDICCIÓN Y GDA
def get_agro_stats(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true&hourly=temperature_2m,relative_humidity_2m&forecast_days=1"
    try:
        r = requests.get(url).json()
        temp = r['current_weather']['temperature']
        hr = r['hourly']['relative_humidity_2m'][0]
        # Cálculo GDA (Base 18.5°C para Monilia/Phyto)
        gda_acumulado = max(0, temp - 18.5) * 15
        return temp, hr, gda_acumulado
    except: return 26.0, 88.0, 105.0

# 5. CONSTRUCCIÓN DEL MAPA (Liviano y Satelital)
# Usamos un mapa base híbrido: Satélite de fondo pero con etiquetas de calles
m = folium.Map(location=[-0.1, -79.5], zoom_start=8, tiles=None)

# Capa Satelital (Fondo tipo Google Earth)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite (Híbrido)',
    overlay=False
).add_to(m)

# Capa de Riesgo GEE (Transparente encima del satélite)
gee_map_id = get_gee_layer()
folium.TileLayer(
    tiles=gee_map_id['tile_fetcher'].url_format,
    attr='Shield-Agro GEE Analysis',
    name='Mapa de Riesgo Fitosanitario',
    overlay=True,
    opacity=0.55
).add_to(m)

# 6. MAPA DE CALOR Y TRAZABILIDAD
heat_data = [[row['lat'], row['lon'], 0.9] for _, row in sensores_pro.iterrows()]
plugins.HeatMap(heat_data, radius=40, blur=25, name="Zonas de Presión Fúngica").add_to(m)

for _, row in sensores_pro.iterrows():
    temp, hr, gda = get_agro_stats(row['lat'], row['lon'])

    # Lógica de Riesgos
    riesgo_m = "ALTO" if gda > 100 else "BAJO"
    riesgo_p = "CRÍTICO" if hr > 90 else "NORMAL"
    color_marker = "#e74c3c" if riesgo_m == "ALTO" or riesgo_p == "CRÍTICO" else "#27ae60"

    html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; width: 260px; border-radius: 10px; border: 1px solid #ccc; background: white;">
        <div style="background: {color_marker}; color: white; padding: 10px; border-radius: 10px 10px 0 0; text-align: center;">
            <b style="font-size: 14px;">🛡️ SHIELD-AGRO: {row['lote']}</b>
        </div>
        <div style="padding: 12px; font-size: 12px; color: #333;">
            <b>📍 Georreferencia:</b> {row['lat']}, {row['lon']}<br>
            <b>🌱 Variedad:</b> {row['variedad']}<br>
            <hr style="border: 0.5px solid #eee;">
            <b>🌡️ Temp:</b> {temp}°C | <b>💧 HR:</b> {hr}%<br>
            <b>📈 GDA Acumulados:</b> {gda:.1f} °D<br>
            <hr style="border: 0.5px solid #eee;">
            <b style="color: #c0392b;">🍄 Riesgo Monilia:</b> {riesgo_m}<br>
            <b style="color: #2980b9;">💧 Riesgo Phytophthora:</b> {riesgo_p}<br>
            <hr style="border: 0.5px solid #eee;">
            <b>👤 Responsable:</b> {row['responsable']}<br>
            <div style="text-align: center; margin-top: 10px;">
                <a href="https://www.google.com/maps?q={row['lat']},{row['lon']}" target="_blank"
                   style="background: #2c3e50; color: white; padding: 5px 10px; text-decoration: none; border-radius: 5px;">Ir al Lote</a>
            </div>
        </div>
    </div>
    """

    folium.RegularPolygonMarker(
        location=[row['lat'], row['lon']],
        number_of_sides=4,
        radius=14,
        rotation=45,
        color='#2c3e50',
        fill_color=color_marker,
        fill_opacity=0.9,
        popup=folium.Popup(html, max_width=300),
        tooltip=f"Ver Trazabilidad: {row['lote']}"
    ).add_to(m)

# 7. ELEMENTOS FINALES
folium.LayerControl(collapsed=False).add_to(m)
m.save("index.html")
print("✅ ¡Plataforma Shield-Agro generada! Aspecto satelital GEE con trazabilidad interactiva.")
m



/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/S2_SR_HARMONIZED

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


✅ ¡Plataforma Shield-Agro generada! Aspecto satelital GEE con trazabilidad interactiva.
